In [1]:
import sys
from pathlib import Path
root = Path().resolve().parent  # adjust level as needed
sys.path.insert(0, str(root))

In [28]:
import pickle
with open(root / "outputs" / "SPRI_ALL_split_documents.pkl", "rb") as f:
    split_documents = pickle.load(f)

In [29]:
len(split_documents)

248

In [4]:
split_documents[-1].page_content

"문서의 표제/요약 섹션: 'AI Index 2025의 주요 내용 및 시사점' 제목과 영어 부제, 발행기관(글로벌 R&D 센터) 주소 및 웹사이트를 포함한 표지·요약 메타데이터\n\nAI Index 2025의 주요 내용 및 시사점\n\n\n  \n\nSummary and Implications of 2025 AI Index Report  \n경기도 성남시 분당구 대왕판교로 712번길 22 글로벌 R&D 연구동(B) 4층\n\nGlobal R&D Center 4F 22 Daewangpangyo-ro 712beon-gil, Bundang-gu, Seongnam-si, Gyeonggi-do\n\n  \n\nwww.spri.kr"

In [5]:
from langchain_upstage import UpstageEmbeddings

In [6]:
from reranker.rrf import ReciprocalRankFusion

In [31]:
from langchain_community.retrievers import BM25Retriever
embeddings = UpstageEmbeddings(model="embedding-passage")
bm25_retriever = BM25Retriever.from_documents(split_documents)
bm25_retriever.k = 10

In [8]:
from langchain_community.vectorstores import FAISS


In [30]:
vectorstore = FAISS.load_local(
        root / "faiss_index", 
        embeddings,
        "SPRI_ALL_contextual",
        allow_dangerous_deserialization=True  # needed in newer versions
    )
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [32]:
def retrieve_document(question: str) -> list[str]:
    retrieved_docs_faiss = faiss_retriever.invoke(question)
    retrieved_docs_bm25 = bm25_retriever.invoke(question)
    retrieved_docs_faiss = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_faiss)
    retrieved_docs_bm25 = ReciprocalRankFusion.calculate_rank_score(retrieved_docs_bm25)
    retrieved_docs = retrieved_docs_faiss + retrieved_docs_bm25
    rrf_docs = ReciprocalRankFusion.get_rrf_docs(retrieved_docs, cutoff=10)
    return rrf_docs

In [11]:
import pandas as pd

In [33]:
df = pd.read_csv(root / "outputs" / "SPRI_ALL_synthetic_single_chunk.csv")

In [34]:
len(df)

200

In [35]:
retrieved_docs = [retrieve_document(df.iloc[i]["query"]) for i in range(len(df))]

In [36]:
def recall(df: pd.DataFrame, retrieved_docs: list[str]) -> dict:
    true_positives = 0
    false_negatives = 0

    for i, row in df.iterrows():
        # 중복 페이지 제거
        # reference_page_number = list({int(page) for page in row["page_number"].strip("[]").split(",")})
        reference_chunk_id = row["chunk_id"]
        retrieved_chunk_id = [doc.metadata["chunk_id"] for doc in retrieved_docs[i]]
       
        if reference_chunk_id in retrieved_chunk_id:
            true_positives += 1
        else:
            print("index: ", i)
            print(row["query"])
            print(row["chunk_id"])
            print(retrieved_chunk_id)
            false_negatives += 1

    print(f"True Positives: {true_positives}, False Negatives: {false_negatives}")

    recall = true_positives / (true_positives + false_negatives)
    return {"recall": recall}

In [37]:
score = recall(df, retrieved_docs)

index:  53
연구 분야에서 미·중의 주도적 위치와 중국의 출판량·인용수 추세, 대학과 산업의 연구 주체별 역할은 어떻게 나타나나요?
page_51_chunk_2
['page_110_chunk_2', 'page_54_chunk_2', 'page_10_chunk_1', 'page_9_chunk_1', 'page_55_chunk_2', 'page_71_chunk_2', 'page_55_chunk_1', 'page_129_chunk_1', 'page_7_chunk_2', 'page_110_chunk_1']
index:  116
연구분야에서 중국의 출판량·인용수는 미국과 비교해 어떤가요, 그리고 대학과 산업 중 누가 새로운 모델 개발을 더 주도하나요?
page_51_chunk_2
['page_55_chunk_2', 'page_110_chunk_2', 'page_36_chunk_1', 'page_55_chunk_1', 'page_9_chunk_1', 'page_10_chunk_1', 'page_110_chunk_1', 'page_130_chunk_2', 'page_54_chunk_2', 'page_113_chunk_1']
True Positives: 198, False Negatives: 2


In [40]:
for doc in split_documents:
    if doc.metadata['chunk_id'] == 'page_51_chunk_2':
        print(doc.id)
        print(doc.page_content)
        print(doc.metadata)
        print("-"*100)

None
SPRi 이슈리포트 IS-159 「AI Index 2023의 주요 내용 및 시사점」 문서의 I. AI Index 2023 주요내용(1.1 보고서 개요 및 특징)에 포함된 ‘주요 내용 요약’ 표로, 연구·기술성능·AI윤리·투자·교육·정책·다양성·여론 등 분야별 핵심 요약을 제시한다.

![](/images/SPRI_2023_cropped_table_41.png)

| 분야 | 내용 |
| --- | --- |
| 연구분야 | § 연구 분야에서는 미·중이 주도하고 있으며, 중국이 출판량 및 인용수측면에서 미국을 추월하고 있는 가운데 양국 간 협력 연구 증가세 지속, 대학은 가장 활발한 연구주체이나 새로운 모델 개발은 산업이 학계를 앞서는 중 |
| 기술성능 | § 기술 측면에서 AI 성능 향상은 다양한 벤치마크 테스트 기준으로 정점에 도달했으며 지속적 성능 향상 유지, 하드웨어의 고성능·저비용화로 연구 장벽이 완화되는 추세 |
| AI윤리 | § AI 윤리 이슈관련 사례증가, AI의 오남용 방지, 공정성, 투명성 관련 연구 및 지표 증가 |
| AI투자 | § 글로벌 경기 침체의 영향으로 AI 분야 투자는 2013년 이후 처음 감소세로 전환했으나 AI관련 기업에 대한 투자는 여전히 높은 편이며, 전 세계적으로 AI 직군의 인력 수요도 높은 상태 |
| AI교육 | § 기존 고등교육 위주의 AI교육이 초중등 교육 과정으로 점차 보편화 되는 추세이며, 다양한 국가로 확산 중 |
| 정책 | § 2019년을 정점으로 AI 국가 전략 발표는 감소하고 있으나 AI확산과 더불어 AI 관련 소송과 각국의 AI 관련 입법이 크게 증가 |
| 다양성 | § AI 전문 연구 인력 관련 성, 인종 등 다양성 측면에서 여전한 격차가 있으나 완화 중 |
| 여론 | § AI인식에 대한 개인적, 국가별 차이가 있으나 대체로 긍정 의견이 부정 의견보다 우세 |

1) Standford HAI(Stanford Institute for Human Centered Artificial I

In [39]:
score

{'recall': 0.99}

In [41]:
def sample_data(df: pd.DataFrame, sample_size: int = 100) -> pd.DataFrame:
    return df.sample(n=sample_size)

In [42]:
len(df)

200

In [43]:
df_sample = sample_data(df)

In [44]:
len(df_sample)

100

In [46]:
df.head()

,query,answer,chunk_id
0,미국의 공공 부문 AI R&D에서 2018년 비국방 예산과 2022년 비교는 얼마이...,"- 2018년 비국방 AI 예산은 5.6억 달러였고, 2022년에는 17.3억 달러...",page_86_chunk_1
1,"VQA의 2015년과 2021년 성능과 인간 수준은 어떻게 변화했으며, MTV가 K...",VQA 성능은 2015년에 55.4%에서 2021년에 79.8%로 상승했으며 인간 ...,page_17_chunk_1
2,AI Index 2022에서 보고된 LLM 성능 변화와 편향 관련 주요 사실은 무엇...,AI Index 2022의 주요 시사점으로는 LLM 성능이 향상되는 동시에 편향(b...,page_5_chunk_3
3,"문서에 표시된 보고서 제목과 장 이름은 무엇이며, 이어서 시작되는 참고문헌 섹션은 ...",보고서 제목은 'SPRi 이슈리포트 IS-159'이고 장 이름은 'AI Index ...,page_102_chunk_1
4,"2016년과 2021년의 AI 관련 법안 수는 각각 얼마이며, 유럽의 인공지능법(안...","AI 관련 법안은 2016년에 1건에서 2021년에 18건으로 증가했으며, 유럽의 ...",page_38_chunk_1


In [45]:
df_sample.head()

,query,answer,chunk_id
3,"문서에 표시된 보고서 제목과 장 이름은 무엇이며, 이어서 시작되는 참고문헌 섹션은 ...",보고서 제목은 'SPRi 이슈리포트 IS-159'이고 장 이름은 'AI Index ...,page_102_chunk_1
198,"2024년에 발표된 AI 모델 중 구글이 개발하거나 출시한 모델들은 무엇이며, 각 ...",구글 관련 모델은 다음 세 가지입니다.\n\n1) FireSat (`24.9) — ...,page_122_chunk_2
163,"2023년과 2024년에 AI 규정을 발표한 기관 수는 각각 몇 곳이며, 보건복지부...","AI 규정을 발표한 기관 수는 2023년 21곳, 2024년 42곳이며, 보건복지부...",page_124_chunk_1
129,"이 표는 무엇을 비교하며, 비교 대상 25개국에 한국과 스웨덴이 포함되어 있나요?","이 표는 ‘입법 과정에서 AI 언급 빈도(25개국 대상)’를 비교하며, 비교 대상 ...",page_33_chunk_2
174,스탠포드 Aviary 프로젝트가 제시한 세 가지 과학 난제는 무엇인가요?,"분자 복제를 위한 DNA 조작, 과학 논문 기반 질의응답, 단백질 안정성 설계",page_120_chunk_2
